<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_12_recurrent_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12: Neural Networks That Write Like Shakespeare
### Recurrent Layers for Variable-Length Data

Standard neural networks require fixed-size inputs. Recurrent Neural Networks (RNNs) handle variable-length sequences by maintaining a hidden state that is updated at each time step.

## 1. The Limitation of Bag-of-Words

A bag-of-words vector loses word order. 'The dog bit the man' and 'The man bit the dog' have the same bag-of-words representation but opposite meanings.

In [ ]:
import numpy as np

vocab    = ["the", "dog", "bit", "man"]
word2idx = {w: i for i, w in enumerate(vocab)}

def bag_of_words(sentence):
    vec = np.zeros(len(vocab))
    for w in sentence:
        if w in word2idx:
            vec[word2idx[w]] += 1
    return vec

s1 = ["the", "dog", "bit", "the", "man"]
s2 = ["the", "man", "bit", "the", "dog"]

b1 = bag_of_words(s1)
b2 = bag_of_words(s2)

print(f"Sentence 1: {' '.join(s1)}")
print(f"Bag-of-words: {b1}")
print()
print(f"Sentence 2: {' '.join(s2)}")
print(f"Bag-of-words: {b2}")
print()
print(f"Identical? {np.array_equal(b1, b2)}")
print("Bag-of-words loses word order entirely.")

## 2. The RNN Architecture

At each time step t, an RNN combines the current input with the previous hidden state to produce a new hidden state. The same weights are shared across all time steps.

```
h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b)
```

In [ ]:
import numpy as np

np.random.seed(0)

input_size  = 4
hidden_size = 3

W_xh = np.random.rand(input_size,  hidden_size) - 0.5
W_hh = np.random.rand(hidden_size, hidden_size) - 0.5
b    = np.zeros(hidden_size)

def rnn_step(x_t, h_prev):
    return np.tanh(x_t.dot(W_xh) + h_prev.dot(W_hh) + b)

sequence = [
    np.array([1, 0, 0, 0]),
    np.array([0, 1, 0, 0]),
    np.array([0, 0, 1, 0]),
    np.array([0, 0, 0, 1]),
]

h = np.zeros(hidden_size)

print(f"{'Step':<6} {'Hidden State'}")
print("-" * 40)
for t, x in enumerate(sequence):
    h = rnn_step(x.astype(float), h)
    print(f"t={t}    {np.round(h, 4)}")

## 3. Forward Propagation Through a Sentence

The RNN processes one word at a time, updating the hidden state at each step. The final hidden state encodes the full sentence.

In [ ]:
import numpy as np

np.random.seed(1)

vocab     = ["the", "dog", "bit", "man", "cat", "sat"]
word2idx  = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
embed_dim  = 4
hidden_size = 5

W_embed = np.random.rand(vocab_size, embed_dim) - 0.5
W_xh    = np.random.rand(embed_dim,  hidden_size) - 0.5
W_hh    = np.random.rand(hidden_size, hidden_size) - 0.5

sentence = ["the", "dog", "bit", "man"]

h = np.zeros(hidden_size)
print("Processing sentence:", ' '.join(sentence))
print()
for word in sentence:
    x = W_embed[word2idx[word]]
    h = np.tanh(x.dot(W_xh) + h.dot(W_hh))
    print(f"After '{word}': h = {np.round(h, 3)}")

print(f"\nFinal sentence vector (last h): {np.round(h, 3)}")

## 4. Character-Level Language Model

An RNN can be trained to predict the next character given all previous characters — a character-level language model. After training, it can generate text character by character.

In [ ]:
import numpy as np

np.random.seed(42)

text = "hello world"
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)
hidden_size = 8
alpha = 0.01

W_xh = np.random.rand(vocab_size,  hidden_size) * 0.1
W_hh = np.random.rand(hidden_size, hidden_size) * 0.1
W_hy = np.random.rand(hidden_size, vocab_size)  * 0.1

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def one_hot(idx, size):
    v = np.zeros(size)
    v[idx] = 1
    return v

print("Characters in vocabulary:", chars)
print("Vocabulary size:", vocab_size)
print()

total_loss = 0
h = np.zeros(hidden_size)
for t in range(len(text) - 1):
    x = one_hot(char2idx[text[t]], vocab_size)
    y_true = char2idx[text[t + 1]]

    h    = np.tanh(x.dot(W_xh) + h.dot(W_hh))
    logit = h.dot(W_hy)
    prob  = softmax(logit)

    loss = -np.log(prob[y_true] + 1e-8)
    total_loss += loss

    print(f"Input: '{text[t]}' -> Predicted: '{idx2char[prob.argmax()]}' | True: '{text[t+1]}' | Loss: {loss:.3f}")

print(f"\nTotal loss (untrained): {total_loss:.3f}")

## 5. The Vanishing Gradient Problem

As backpropagation flows through many time steps, gradients get multiplied by weights repeatedly. If those weights are < 1, the gradient shrinks exponentially (vanishes). This prevents RNNs from learning long-range dependencies.

In [ ]:
import numpy as np

gradient = 1.0

print("Gradient magnitude over time steps:")
print()
print(f"{'Steps':>6} {'Small W (0.5)':>16} {'Large W (2.0)':>16}")
print("-" * 42)

g_small = 1.0
g_large = 1.0
for t in range(1, 11):
    g_small *= 0.5
    g_large *= 2.0
    print(f"{t:>6} {g_small:>16.6f} {g_large:>16.2f}")

print("\nSmall weights -> vanishing gradient")
print("Large weights -> exploding gradient")